# Lecture 3c — Light Yield & Event Displays with **Prometheus**

### Neutrino Interactions, Simulation and Event Generation · *N. Kamp*

This is part of a **3-notebook** set following the simulation pipeline:
```
  FLUX  ->  INTERACTION  ->  PROPAGATION  ->  LIGHT + DETECTOR
  [ SIREN tables + injection + weights ]  [ PROPOSAL ]  [ Prometheus ]
     part a  (this set: a=SIREN, b=PROPOSAL, c=Prometheus)
```

**This notebook (part c)** is the payoff: from the charged particles (3a/3b) to **Cherenkov photons**
to **what the detector sees** — and the $\nu_\tau$ **double-bang** event display.

## Setup

Run this first. On Colab it clones the repo so `helpers` and `data/` exist, then puts `src/` on the
path. **Set `REPO_URL` to your repository.** Every data stage reads a **real** file you generated
with `make_cache.py` (committed to `data/` or hosted); if one is missing, the cell raises a clear
error telling you which `make_cache.py` command to run. There are no synthetic stand-ins.

In [ ]:
# === Setup: make the repo's helper code importable (the Colab fix) ===
# Colab's GitHub browser loads ONLY this .ipynb -- the surrounding repo
# (src/helpers.py, data/) is NOT cloned. So we clone it into the runtime here.
import os, sys, subprocess

REPO_URL = 'https://github.com/USER/REPO.git'   # <-- set to your repo
SUBDIR   = 'Lecture3_Simulation'                # folder that holds src/ and data/

if 'google.colab' in sys.modules and not os.path.isdir('REPO'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, 'REPO'], check=True)

# Find the folder containing src/helpers.py, whether cloned (Colab) or in-repo (local):
_cands = ['REPO/' + SUBDIR, SUBDIR, '.', '..', '../..']
_base  = next((p for p in _cands if os.path.isdir(os.path.join(p, 'src'))), None)
assert _base, 'Could not find src/ -- check REPO_URL / SUBDIR.'
sys.path.insert(0, os.path.abspath(os.path.join(_base, 'src')))

def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])
if 'google.colab' in sys.modules:
    pip('pyarrow', 'awkward')        # light deps for reading cached data
    # prometheus + proposal compile from source; we DON'T build them live.
    # We display a cached Prometheus output instead (see make_cache.py).

import numpy as np, matplotlib.pyplot as plt
import helpers as H
print('helpers loaded from:', H.__file__)
for _m in ('siren', 'proposal', 'prometheus'):     # did the installs work?
    print(f'  {_m:11s} importable: {H.have(_m)}')

> **Running on Colab.** The disk is wiped each session, so re-run Setup every time. The data plotted
> below is always **real**: either produced by the tool live (if installed) or read from a cached
> file you made with `make_cache.py`. SIREN/PROPOSAL/Prometheus are compiled C++; SIREN can install
> from a **prebuilt wheel** (see README), while PROPOSAL/Prometheus are slow to build — so for those
> we read cached output. A missing cache raises a clear error (it never invents data).

## 1. Light yield + detector response — run Prometheus, then *see* the event

**Prometheus** is the open-source telescope simulation: it takes the injected event, propagates the
leptons (PROPOSAL), generates Cherenkov light, and propagates photons to the modules. The full
IceCube chain uses **PPC** (GPU); the open-source CPU path uses a **parametric photon yield**
(`fennel`) — slower per photon but no GPU needed, perfect for Colab.

Because Prometheus (via PROPOSAL) **compiles from source**, we **don't build it live** — we load a
cached Prometheus output and display it. The config below is shown so you see how the pieces connect;
if the cached file is missing, the loader raises a clear error (run `make_cache.py --events`).

In [ ]:
# ---- How a Prometheus nu_tau run is configured (illustrative) ----
# Run this in make_cache.py (full environment), NOT live in Colab.
if H.have('prometheus'):
    from prometheus import Prometheus
    from prometheus.config import config
    config['run']['nevents'] = 50
    config['injection']['name'] = 'SIREN'              # or 'LeptonInjector'
    config['detector']['geo file'] = '<path>/icecube.geo'
    config['photon propagator']['name'] = 'olympus'    # CPU path; avoids GPU PPC
    # p = Prometheus(config); p.sim()   # writes a parquet of photon hits + truth
    print('Prometheus configured (CPU photon propagator). See make_cache.build_prometheus_events.')
else:
    print('prometheus not installed -- loading the cached event below.')

In [ ]:
# Load a real cached Prometheus event (make_cache.py --events). Raises a clear
# error if missing -- nothing here is fabricated.
hits, geo = H.load_prometheus_event('prometheus_nutau_example.parquet', event=0)
print('loaded event with', len(hits['x']), 'photon hits')

In [ ]:
ax = H.plot_event_display(hits, geo=geo,
                          title=r'$\nu_\tau$ CC double-bang  (time-coloured photon hits)')
plt.show()

### The full zoo of signatures (Lecture 2 callback)

Same machinery, different final states → different topologies. With real Prometheus output you can
render each of these from truth-labelled events:

| Signature | Origin | Energy reco quality |
|---|---|---|
| **Single cascade** | $\nu_e$ CC, all-NC | best (calorimetric, contained) |
| **Through-going track** | $\nu_\mu$ CC, muon exits | poor (only $dE/dx$ lower bound) |
| **Starting track** | $\nu_\mu$ CC inside | good (cascade + track) |
| **Double bang / double pulse** | $\nu_\tau$ CC | good if both bangs contained |
| **Dimuon** | charm / di-muon | specialised |

> **🔧 Try it.** Pass a different `event=` index to `H.load_prometheus_event` to step through the
> cached sample. Low-energy taus give one merged cascade (looks like a single $\nu_e$ cascade);
> higher-energy ones separate into two bangs — connect this to the decay-length curve in **3b**.

## Wrap-up

**References:** Prometheus (arXiv:2304.14526), SIREN / LeptonInjector (arXiv:2012.10449),
PROPOSAL (Comput.Phys.Commun. 2019), nuSQuIDS (arXiv:2112.13804), SIREN flux tables
(Zenodo 20129082), Formaggio & Zeller (Rev.Mod.Phys. 2012).

**Go further:**
- Render a $\nu_e$ single cascade and a $\nu_\mu$ through-going track from the same Prometheus sample.
- Put the **3b** decay-length curve and this display side by side: predict the topology before plotting.
- Colour hits by charge (`npe`) instead of time to see where the energy is deposited.